In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("../config/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 21))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [ ]:
from scripts.beamforming import get_best_beam, filter_best_beams
from scripts.utils import dataset_tp_rp_split, extract_unique_npcis
from scripts.weighted_coverage import wknn
from scripts.matrix_operations import compute_weights, create_point_matrix
import pandas as pd

df = df_orig.sample(200)

df["best_beam"] = df["measurements_matrix"].apply(
    lambda x: get_best_beam(x, rf_param)
)

df_tp, df_rp = dataset_tp_rp_split(df, 0.3, 12354)

unique_pcis = extract_unique_npcis(df['measurements_matrix'])

m_rp, idx_rp = create_point_matrix(df_rp, unique_pcis, rf_param)

errors = []
for _, row in df.iterrows():
    tp = pd.DataFrame([row])

    tp.loc[:, "measurements_matrix"] = tp.loc[:, "measurements_matrix"].apply(
        lambda x: filter_best_beams(x, rf_param, n_best_pcis=1, use_sidelobes=False)
    )

    m_tp, idx_tp = create_point_matrix(tp, unique_pcis, rf_param)
    W, idx_sort = compute_weights(m_rp, idx_rp, m_tp, idx_tp)
    _, err = wknn(tp, df_rp, idx_sort, W, 1)

    errors.extend(err)

errors = np.array(errors)
print(errors)
print(f'MEAN {errors.mean():.2f}')

